In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

app_cols = [
    'SK_ID_CURR', 'TARGET', 'NAME_CONTRACT_TYPE', 'CODE_GENDER',
    'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'CNT_FAM_MEMBERS',
    'AMT_INCOME_TOTAL', 'AMT_CREDIT', 'AMT_ANNUITY', 'AMT_GOODS_PRICE',
    'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS',
    'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'ORGANIZATION_TYPE',
    'DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION',
    'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'
]

bureau_cols = [
    'SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_TYPE',
    'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT'
]

prev_cols = [
    'SK_ID_CURR', 'SK_ID_PREV', 'NAME_CONTRACT_TYPE', 'NAME_CONTRACT_STATUS',
    'AMT_APPLICATION', 'AMT_CREDIT', 'DAYS_DECISION', 'NAME_CLIENT_TYPE'
]

df_app = pd.read_csv('../data/raw/HC_application_train.csv', usecols=app_cols)
df_bureau = pd.read_csv('../data/raw/HC_bureau.csv', usecols=bureau_cols)
df_prev = pd.read_csv('../data/raw/HC_previous_application.csv', usecols=prev_cols)

print("Data mentah berhasil dimuat untuk proses pembersihan.")

Data mentah berhasil dimuat untuk proses pembersihan.


In [ ]:
df_clean = df_app.copy()
initial_rows = len(df_clean)

df_clean = df_clean[df_clean['CODE_GENDER'] != 'XNA'].copy()

extreme_income = df_clean[df_clean['AMT_INCOME_TOTAL'] > 10_000_000]
print(f"Outlier pendapatan ekstrem (> $10M) ditemukan: {len(extreme_income)} baris (Max: ${df_clean['AMT_INCOME_TOTAL'].max():,.0f})")
df_clean = df_clean[df_clean['AMT_INCOME_TOTAL'] <= 10_000_000].copy()

df_clean['is_pensioner_or_unemployed'] = (df_clean['DAYS_EMPLOYED'] == 365243).astype(int)
df_clean['DAYS_EMPLOYED'] = df_clean['DAYS_EMPLOYED'].replace(365243, np.nan)

df_clean['age_years'] = (df_clean['DAYS_BIRTH'] / -365.25).round(1)
df_clean['employment_years'] = (df_clean['DAYS_EMPLOYED'] / -365.25).round(1)
df_clean['registration_years'] = (df_clean['DAYS_REGISTRATION'] / -365.25).round(1)

df_clean = df_clean.drop(columns=['DAYS_BIRTH', 'DAYS_EMPLOYED', 'DAYS_REGISTRATION'])

df_clean['OCCUPATION_TYPE'] = df_clean['OCCUPATION_TYPE'].fillna('Unknown / Not Specified')

df_clean['AMT_ANNUITY'] = df_clean['AMT_ANNUITY'].fillna(
    df_clean.groupby('NAME_INCOME_TYPE')['AMT_ANNUITY'].transform('median')
)
df_clean['CNT_FAM_MEMBERS'] = df_clean['CNT_FAM_MEMBERS'].fillna(df_clean['CNT_FAM_MEMBERS'].median())

df_clean['AMT_GOODS_PRICE'] = df_clean['AMT_GOODS_PRICE'].fillna(df_clean['AMT_CREDIT'])

for col in ['EXT_SOURCE_2', 'EXT_SOURCE_3']:
    df_clean[col] = df_clean[col].fillna(
        df_clean.groupby('NAME_INCOME_TYPE')[col].transform('median')
    )

print(f"\nRingkasan Cleaning Tabel Utama:")
print(f"- Baris awal   : {initial_rows:,}")
print(f"- Baris bersih : {len(df_clean):,} (Dihapus: {initial_rows - len(df_clean)} baris anomali/outlier)")
print(f"- Sisa missing values (kecuali EXT_SOURCE_1 & employment_years pensiunan):")
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

Outlier pendapatan ekstrem (> $10M) ditemukan: 3 baris (Max: $117,000,000)

Ringkasan Cleaning Tabel Utama:
- Baris awal   : 307,511
- Baris bersih : 307,504 (Dihapus: 7 baris anomali/outlier)
- Sisa missing values (kecuali EXT_SOURCE_1 & employment_years pensiunan):
EXT_SOURCE_1        173375
EXT_SOURCE_3             5
employment_years     55374
dtype: int64


In [ ]:
df_clean['credit_to_income_ratio'] = (df_clean['AMT_CREDIT'] / df_clean['AMT_INCOME_TOTAL']).round(3)
df_clean['annuity_to_income_ratio'] = (df_clean['AMT_ANNUITY'] / df_clean['AMT_INCOME_TOTAL']).round(4)

df_clean['avg_ext_source'] = df_clean[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1).round(4)

df_clean['age_group'] = pd.cut(
    df_clean['age_years'],
    bins=[18, 25, 35, 45, 55, 100],
    labels=['18-25 (Young)', '26-35 (Early Career)', '36-45 (Mid Career)', '46-55 (Mature)', '55+ (Senior)']
).astype(str)

df_clean['dti_tier'] = pd.cut(
    df_clean['annuity_to_income_ratio'],
    bins=[-1, 0.15, 0.25, 0.35, 100],
    labels=['1. Low (<15%)', '2. Moderate (15-25%)', '3. High Burden (25-35%)', '4. Critical (>35%)']
).astype(str)

df_clean['ext_score_tier'] = pd.cut(
    df_clean['avg_ext_source'],
    bins=[-1, 0.35, 0.55, 1.1],
    labels=['1. High Risk (<0.35)', '2. Medium Risk (0.35-0.55)', '3. Low Risk (>0.55)']
).astype(str)

print("Feature engineering berhasil dibuat! Contoh 5 baris fitur baru:")
display(df_clean[['SK_ID_CURR', 'age_years', 'age_group', 'credit_to_income_ratio', 
                  'annuity_to_income_ratio', 'dti_tier', 'avg_ext_source', 'ext_score_tier']].head())

Feature engineering berhasil dibuat! Contoh 5 baris fitur baru:


,SK_ID_CURR,age_years,age_group,credit_to_income_ratio,annuity_to_income_ratio,dti_tier,avg_ext_source,ext_score_tier
0,100002,25.90,26-35 (Early Career),2.01,0.12,1. Low (<15%),0.16,1. High Risk (<0.35)
1,100003,45.90,46-55 (Mature),4.79,0.13,1. Low (<15%),0.49,2. Medium Risk (0.35-0.55)
2,100004,52.10,46-55 (Mature),2.00,0.10,1. Low (<15%),0.64,3. Low Risk (>0.55)
3,100006,52.00,46-55 (Mature),2.32,0.22,2. Moderate (15-25%),0.58,3. Low Risk (>0.55)
4,100007,54.60,46-55 (Mature),4.22,0.18,2. Moderate (15-25%),0.42,2. Medium Risk (0.35-0.55)


In [ ]:
df_bureau_clean = df_bureau.copy()

df_bureau_clean['AMT_CREDIT_SUM_DEBT'] = df_bureau_clean['AMT_CREDIT_SUM_DEBT'].clip(lower=0).fillna(0)
df_bureau_clean['AMT_CREDIT_SUM'] = df_bureau_clean['AMT_CREDIT_SUM'].clip(lower=0).fillna(0)

df_bureau_clean['is_active'] = (df_bureau_clean['CREDIT_ACTIVE'] == 'Active').astype(int)
df_bureau_clean['is_closed'] = (df_bureau_clean['CREDIT_ACTIVE'] == 'Closed').astype(int)
df_bureau_clean['has_overdue'] = (df_bureau_clean['CREDIT_DAY_OVERDUE'] > 0).astype(int)

agg_bureau = df_bureau_clean.groupby('SK_ID_CURR', as_index=False).agg(
    bureau_total_loans=('SK_ID_BUREAU', 'count'),
    bureau_active_loans=('is_active', 'sum'),
    bureau_closed_loans=('is_closed', 'sum'),
    bureau_total_credit=('AMT_CREDIT_SUM', 'sum'),
    bureau_total_debt=('AMT_CREDIT_SUM_DEBT', 'sum'),
    bureau_max_overdue_days=('CREDIT_DAY_OVERDUE', 'max'),
    bureau_overdue_count=('has_overdue', 'sum')
)

agg_bureau['bureau_has_overdue'] = (agg_bureau['bureau_overdue_count'] > 0).astype(int)

print(f"Agregasi Bureau selesai: {agg_bureau.shape[0]:,} nasabah unik x {agg_bureau.shape[1]} kolom")
display(agg_bureau.head())

Agregasi Bureau selesai: 305,811 nasabah unik x 9 kolom


,SK_ID_CURR,bureau_total_loans,bureau_active_loans,bureau_closed_loans,bureau_total_credit,bureau_total_debt,bureau_max_overdue_days,bureau_overdue_count,bureau_has_overdue
0,100001,7,3,4,"1,453,365.00","596,686.50",0,0,0
1,100002,8,2,6,"865,055.56","245,781.00",0,0,0
2,100003,4,1,3,"1,017,400.50",0.00,0,0,0
3,100004,2,0,2,"189,037.80",0.00,0,0,0
4,100005,3,2,1,"657,126.00","568,408.50",0,0,0


In [5]:
df_prev_clean = df_prev.copy()

df_prev_clean['is_approved'] = (df_prev_clean['NAME_CONTRACT_STATUS'] == 'Approved').astype(int)
df_prev_clean['is_refused'] = (df_prev_clean['NAME_CONTRACT_STATUS'] == 'Refused').astype(int)

agg_prev = df_prev_clean.groupby('SK_ID_CURR', as_index=False).agg(
    prev_app_count=('SK_ID_PREV', 'count'),
    prev_approved_count=('is_approved', 'sum'),
    prev_refused_count=('is_refused', 'sum'),
    prev_avg_application=('AMT_APPLICATION', 'mean'),
    prev_avg_credit=('AMT_CREDIT', 'mean')
)

agg_prev['prev_refusal_rate'] = (agg_prev['prev_refused_count'] / agg_prev['prev_app_count'] * 100).round(2)
agg_prev['prev_avg_application'] = agg_prev['prev_avg_application'].round(2)
agg_prev['prev_avg_credit'] = agg_prev['prev_avg_credit'].round(2)

print(f"Agregasi Previous Application selesai: {agg_prev.shape[0]:,} nasabah unik x {agg_prev.shape[1]} kolom")
display(agg_prev.head())

Agregasi Previous Application selesai: 338,857 nasabah unik x 7 kolom


,SK_ID_CURR,prev_app_count,prev_approved_count,prev_refused_count,prev_avg_application,prev_avg_credit,prev_refusal_rate
0,100001,1,1,0,"24,835.50","23,787.00",0.00
1,100002,1,1,0,"179,055.00","179,055.00",0.00
2,100003,3,3,0,"435,436.50","484,191.00",0.00
3,100004,1,1,0,"24,282.00","20,106.00",0.00
4,100005,2,1,0,"22,308.75","20,076.75",0.00


In [ ]:
df_clean.columns = df_clean.columns.str.lower()
agg_bureau.columns = agg_bureau.columns.str.lower()
agg_prev.columns = agg_prev.columns.str.lower()

ordered_fact_cols = [
    'sk_id_curr', 'target', 'name_contract_type', 'code_gender',
    'flag_own_car', 'flag_own_realty', 'cnt_children', 'cnt_fam_members',
    'amt_income_total', 'amt_credit', 'amt_annuity', 'amt_goods_price',
    'name_income_type', 'name_education_type', 'name_family_status',
    'name_housing_type', 'occupation_type', 'organization_type',
    'ext_source_1', 'ext_source_2', 'ext_source_3',
    'is_pensioner_or_unemployed', 'age_years', 'employment_years',
    'registration_years', 'credit_to_income_ratio', 'annuity_to_income_ratio',
    'avg_ext_source', 'age_group', 'dti_tier', 'ext_score_tier'
]
df_clean = df_clean[ordered_fact_cols]

df_clean.to_csv('../data/processed/fact_applications.csv', index=False)
agg_bureau.to_csv('../data/processed/dim_bureau_history.csv', index=False)
agg_prev.to_csv('../data/processed/dim_prev_applications.csv', index=False)

print("=== EKSPOR ULANG SELESAI ===")
print(f"1. fact_applications.csv : {df_clean.shape[0]:,} baris x {df_clean.shape[1]} kolom")

=== EKSPOR ULANG SELESAI ===
1. fact_applications.csv : 307,504 baris x 31 kolom


: 